<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>05. 🖨️ Custom HTML Template Engine from Scratch</b>
</div>


### 📌 Project Overview
In this project, we build a compiling HTML Template Engine (similar to Jinja2 or Django Templates) from scratch.
Rather than slow replacement string scanning, we write a lexer that translates template structures (variable interpolations `{{ var }}`, iteration loops `{% for x in list %}`, and conditions `{% if cond %}`) into a dynamic Python function code block which we execute on-the-fly using `exec()`.

#### 🔑 Key Concepts Covered:
- Variable tokenization and regex delimiters
- Dynamic Python code construction (managing indentations and control loops programmatically)
- Executing dynamic script blocks using python's built-in `exec()`
- Context namespace resolution in runtime scopes


In [1]:
import re

class HTMLTemplateEngine:
    def __init__(self, template_text):
        self.template_text = template_text
        self.code_lines = []
        self.indent_level = 0
        self.compile_template()
        
    def append_instruction(self, instruction):
        self.code_lines.append('    ' * self.indent_level + instruction)
        
    def compile_template(self):
        # Initialize standard function handler
        self.code_lines = ['def execute_template(context, render_buffer):']
        self.indent_level = 1
        
        # Load context dictionary keys as local variables
        self.append_instruction('for key, val in context.items():')
        self.append_instruction('    globals()[key] = val')
        
        # Split template matching both variables {{...}} and block tags {%...%}
        tokens = re.split(r'({%.*?%}|{{.*?}})', self.template_text)
        
        for token in tokens:
            if not token:
                continue
                
            # Case 1: Variable Substitution
            if token.startswith('{{') and token.endswith('}}'):
                expression = token[2:-2].strip()
                self.append_instruction(f'render_buffer.append(str({expression}))')
                
            # Case 2: Control Blocks
            elif token.startswith('{%') and token.endswith('%}'):
                tag_content = token[2:-2].strip()
                parts = tag_content.split()
                tag_name = parts[0]
                
                if tag_name == 'if':
                    condition = ' '.join(parts[1:])
                    self.append_instruction(f'if {condition}:')
                    self.indent_level += 1
                elif tag_name == 'for':
                    loop_var = parts[1]
                    iterable = ' '.join(parts[3:])  # Skip 'in'
                    self.append_instruction(f'for {loop_var} in {iterable}:')
                    self.indent_level += 1
                elif tag_name in ('endif', 'endfor'):
                    self.indent_level -= 1
                    
            # Case 3: Literal Text / HTML Blocks
            else:
                # Keep quotes safe by calling repr()
                self.append_instruction(f'render_buffer.append({repr(token)})')
                
    def render(self, context_data):
        buffer = []
        compiled_script = '\n'.join(self.code_lines)
        
        local_namespace = {}
        # Compile script dynamically to define execute_template inside namespace
        exec(compiled_script, globals(), local_namespace)
        
        render_fn = local_namespace['execute_template']
        render_fn(context_data, buffer)
        return "".join(buffer)

In [2]:
template_string = '''
<h2>Student Name: {{ name }}</h2>
<p>Selected Course: {{ course }}</p>

{% if score >= 80 %}
  <p style="color: green;">Grade: Excellent Performance (Score: {{ score }})</p>
{% else %}
  <p style="color: orange;">Grade: Passing Performance (Score: {{ score }})</p>
{% endif %}

<h3>Project Milestones Completed:</h3>
<ul>
{% for task in projects %}
  <li>Task Module: {{ task }}</li>
{% endfor %}
</ul>
'''

context_data = {
    'name': 'Faizy',
    'course': 'Python Metaprogramming',
    'score': 85,
    'projects': ['Metaclasses', 'Descriptors', 'Template Engines']
}

engine = HTMLTemplateEngine(template_string)
result_page = engine.render(context_data)
print('--- COMPILED TEMPLATE OUTPUT ---')
print(result_page)

--- COMPILED TEMPLATE OUTPUT ---

<h2>Student Name: Faizy</h2>
<p>Selected Course: Python Metaprogramming</p>


  <p style="color: green;">Grade: Excellent Performance (Score: 85)</p>

  <p style="color: orange;">Grade: Passing Performance (Score: 85)</p>


<h3>Project Milestones Completed:</h3>
<ul>

  <li>Task Module: Metaclasses</li>

  <li>Task Module: Descriptors</li>

  <li>Task Module: Template Engines</li>

</ul>

